# 02 — Prompt Variant Comparison

Where do the five prompt variants agree and disagree? This notebook uses the
confidence scores from `evaluate_confidence.py` and the judge scores from
`evaluate_llm_judge.py` to map the landscape of variant agreement.

Key questions:
- Which variant produces the most divergent outputs?
- Are there language families where a particular variant consistently wins?
- How sensitive is the pipeline to prompt choice? (Sensitivity = the paper's
  core empirical argument about epistemic stance in prompt design)

**Run order:** After `evaluate_confidence.py` and `evaluate_llm_judge.py`.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from data_generation_scripts.utils import get_data_directory_path, read_csv_file

DATA_DIR = get_data_directory_path()
EVAL_DIR = os.path.join(DATA_DIR, 'metadata_files', 'evaluation')
TERMS = ['Computational Humanities', 'Digital Humanities']
VARIANTS = ['comparative', 'minimal', 'expert_persona', 'contextual', 'native_rationale']

VARIANT_LABELS = {
    'comparative':     'Comparative',
    'minimal':         'Minimal',
    'expert_persona':  'Expert Persona',
    'contextual':      'Contextual',
    'native_rationale':'Native Rationale',
}
VARIANT_COLOURS = {
    'comparative':     '#4C72B0',
    'minimal':         '#DD8452',
    'expert_persona':  '#55A868',
    'contextual':      '#C44E52',
    'native_rationale':'#8172B2',
}

# Load evaluation outputs
conf_path  = os.path.join(EVAL_DIR, 'confidence_scores.csv')
judge_path = os.path.join(EVAL_DIR, 'llm_judge_scores.csv')

conf_df  = read_csv_file(conf_path)  if os.path.exists(conf_path)  else None
judge_df = read_csv_file(judge_path) if os.path.exists(judge_path) else None

print('Confidence scores:', len(conf_df) if conf_df is not None else 'NOT FOUND')
print('Judge scores:',      len(judge_df) if judge_df is not None else 'NOT FOUND')

## 2.1 Confidence Score Distribution by Variant

How does primary-service agreement differ across prompt variants?
Note: for non-comparative variants, primary services (GT, EasyNMT) are the same
across variants — this plot shows whether the LLM variant output aligns with
the primary service consensus.

In [ ]:
if conf_df is not None and 'prompt_variant' in conf_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Violin plot: primary confidence by variant
    plot_data = conf_df[conf_df['primary_total_services'] > 0].copy()
    variant_order = [v for v in VARIANTS if v in plot_data['prompt_variant'].unique()]
    
    sns.violinplot(
        data=plot_data, x='prompt_variant', y='primary_confidence',
        order=variant_order, palette=VARIANT_COLOURS,
        inner='quartile', ax=axes[0]
    )
    axes[0].axhline(0.6, color='red', linestyle='--', alpha=0.7, label='threshold (0.6)')
    axes[0].set_title('Primary Service Confidence Distribution by Variant')
    axes[0].set_xlabel('Prompt Variant')
    axes[0].set_ylabel('Confidence Score')
    axes[0].set_xticklabels([VARIANT_LABELS.get(v, v) for v in variant_order], rotation=20, ha='right')
    axes[0].legend()
    
    # Stacked bar: above/below threshold per variant
    threshold_counts = plot_data.groupby('prompt_variant')['primary_above_threshold'].value_counts(normalize=True).unstack(fill_value=0)
    if True in threshold_counts.columns:
        threshold_counts[True].reindex(variant_order).plot(
            kind='bar', ax=axes[1], color=[VARIANT_COLOURS.get(v, 'grey') for v in variant_order],
            edgecolor='white'
        )
    axes[1].set_title('% Above Confidence Threshold (0.6) by Variant')
    axes[1].set_ylabel('Proportion')
    axes[1].set_xticklabels([VARIANT_LABELS.get(v, v) for v in variant_order], rotation=20, ha='right')
    axes[1].set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, '02_confidence_by_variant.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Confidence scores not available or missing prompt_variant column')

## 2.2 Inter-Variant Agreement Matrix

For each pair of variants, what proportion of languages produce identical translations?
High pairwise agreement → prompt choice doesn't matter for that language pair.
Low pairwise agreement → the epistemic stance embedded in the prompt shapes the output.

In [ ]:
# Load per-variant term outputs and compute pairwise agreement
def load_variant_terms(data_dir, term, variants=VARIANTS):
    term_slug = term.lower().replace(' ', '_')
    dfs = {}
    for v in variants:
        if v == 'comparative':
            path = os.path.join(data_dir, 'metadata_files', 'translated_terms',
                                term_slug, 'initial_translated_terms.csv')
        else:
            path = os.path.join(data_dir, 'metadata_files', 'translated_terms',
                                term_slug, 'prompt_variants', f'{v}_initial_translated_terms.csv')
        if os.path.exists(path):
            df = read_csv_file(path)
            if 'term' in df.columns:
                dfs[v] = df.set_index('language_code')['term']
    return dfs

for term in TERMS:
    variant_terms = load_variant_terms(DATA_DIR, term)
    if len(variant_terms) < 2:
        print(f'{term}: insufficient variant data')
        continue
    
    # Build agreement matrix
    variants_present = list(variant_terms.keys())
    n = len(variants_present)
    matrix = np.zeros((n, n))
    
    for i, v1 in enumerate(variants_present):
        for j, v2 in enumerate(variants_present):
            common = variant_terms[v1].index.intersection(variant_terms[v2].index)
            if len(common) == 0: continue
            agree = (variant_terms[v1][common].fillna('') == 
                     variant_terms[v2][common].fillna('')).mean()
            matrix[i, j] = agree
    
    labels = [VARIANT_LABELS.get(v, v) for v in variants_present]
    fig, ax = plt.subplots(figsize=(7, 6))
    mask = np.zeros_like(matrix, dtype=bool)
    mask[np.triu_indices_from(mask, k=1)] = True  # upper triangle
    sns.heatmap(matrix, annot=True, fmt='.2f', xticklabels=labels, yticklabels=labels,
                cmap='RdYlGn', vmin=0, vmax=1, ax=ax, mask=mask,
                cbar_kws={'label': 'Agreement rate'})
    ax.set_title(f'Inter-variant Agreement — {term}\n(proportion of languages producing identical translations)')
    plt.tight_layout()
    fname = f'02_agreement_matrix_{term.lower().replace(" ","_")}.png'
    plt.savefig(os.path.join(EVAL_DIR, fname), dpi=150, bbox_inches='tight')
    plt.show()

## 2.3 LLM Judge Variant Preferences

Which variant does the LLM judge prefer? Is there a consistent winner,
or does the preferred variant depend on language family?

In [ ]:
if judge_df is not None and 'best_variant' in judge_df.columns:
    # Overall variant win rates
    win_counts = judge_df['best_variant'].value_counts()
    print('LLM Judge — variant win counts:')
    print(win_counts.to_string())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Win rate bar chart
    win_pct = (win_counts / len(judge_df) * 100).reindex(VARIANTS, fill_value=0)
    colours = [VARIANT_COLOURS.get(v, 'grey') for v in win_pct.index]
    win_pct.plot(kind='bar', ax=axes[0], color=colours, edgecolor='white')
    axes[0].set_title('Judge Preferred Variant (% of evaluated rows)')
    axes[0].set_xticklabels([VARIANT_LABELS.get(v, v) for v in win_pct.index],
                             rotation=20, ha='right')
    axes[0].set_ylabel('% of rows')
    
    # Score distribution by variant — parse JSON scores
    score_rows = []
    for _, row in judge_df.iterrows():
        try:
            scores = json.loads(str(row.get('scores_json', '{}')))
            for v, s in scores.items():
                score_rows.append({'variant': v, 'score': float(s),
                                   'language_code': row.get('language_code', '')})
        except Exception:
            pass
    
    if score_rows:
        scores_df = pd.DataFrame(score_rows)
        scores_df['variant_clean'] = scores_df['variant'].str.split('_').str[0]
        sns.boxplot(data=scores_df, x='variant_clean', y='score',
                    palette=VARIANT_COLOURS, ax=axes[1])
        axes[1].set_title('Judge Score Distribution by Variant')
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20, ha='right')
        axes[1].set_ylabel('Judge Score (0–1)')
    
    plt.tight_layout()
    plt.savefig(os.path.join(EVAL_DIR, '02_judge_variant_preferences.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Judge scores not available — run evaluate_llm_judge.py first')

## 2.4 High-Disagreement Cases: Detailed Examples

Pull the cases where variants disagree most strongly and examine them closely.
These are the best candidates for the paper's qualitative analysis.

In [ ]:
if judge_df is not None and 'score_gap' in judge_df.columns:
    # High disagreement = judge had strong preference for one variant
    high_disagree = judge_df[
        judge_df['judge_agrees'] == True
    ].sort_values('score_gap', ascending=False).head(20)
    
    print(f'Top 20 high-disagreement cases (judge strongly preferred one variant):')
    display_cols = ['language_code', 'language_name', 'term_source',
                    'best_variant', 'best_score', 'score_gap', 'concept_exists']
    available = [c for c in display_cols if c in high_disagree.columns]
    print(high_disagree[available].to_string(index=False))
    
    print('\n--- Detailed look at top 5 ---')
    for _, row in high_disagree.head(5).iterrows():
        print(f"\nLanguage: {row.get('language_name', row.get('language_code', '?'))}")
        print(f"Term: {row.get('term_source', '?')}")
        print(f"Best variant: {row.get('best_variant', '?')} (score: {row.get('best_score', '?')})")
        print(f"Score gap: {row.get('score_gap', '?')}")
        print(f"Concept exists: {row.get('concept_exists', '?')}")
        try:
            reasoning = json.loads(str(row.get('reasoning_json', '{}')))
            for v, r in reasoning.items():
                if r: print(f"  [{v}]: {r[:120]}")
        except Exception:
            pass
else:
    print('Judge scores not available')